# Cleaning

## Global Variables

In [1]:
import glob
import pandas as pd
import os

folder_file_path = "Data/Ecom Owners 100k - Copy"
files = glob.glob(folder_file_path + "/*.csv")

## Delete "Unnamed" Columns

## Current Functions

In [ ]:
import pandas as pd
import glob
from collections import defaultdict

"""
This script processes all CSV files in a given directory, identifies and drops
columns that are entirely NaN (Not a Number) or contain 'unnamed' in their name.
It then groups the files based on the specific set of columns that were dropped
from each file and reports these groupings to the console. Files that encounter
read errors are also reported separately.

Requires:
- pandas library
- 'folder_file_path' variable defined with the path to the directory containing CSVs.

Outputs:
- Prints processing progress, error messages for failed files,
  groups of files based on dropped columns, and a summary of files with read errors.
"""

'''
TODO:
- update to display 5 actual values from columns NAN occurs
'''

# Find files
files = glob.glob(folder_file_path + "/*.csv")

if not files:
    print(f"No CSV files found in '{folder_file_path}'. Please check the path.")
else:
    print(f"Found {len(files)} CSV files. Starting processing...")

    # Dictionary to hold the groups: {tuple_of_dropped_columns: [list_of_files]}
    dropped_column_groups = defaultdict(list)
    # Dictionary to keep track of files that cause errors
    read_errors = {}
    processed_count = 0

    # --- Process Files and Group Them ---
    for file in files:
        try:
            # Minimal print during processing loop for progress indication
            # print(f"Processing: {file}")
            df = pd.read_csv(file, low_memory=False)

            original_columns = df.columns.tolist()

            # Drop columns where *ALL* values are NaN
            # Note: Your code used how='all'. If you meant *ANY* NaN, change to how='any'
            df_cleaned = df.dropna(axis=1, how='all')

            # Drop columns with 'unnamed' in the name (usually index columns)
            columns_to_drop = [col for col in df.columns if 'unnamed' in col.lower()]
            df_cleaned.drop(columns=columns_to_drop, axis=1, inplace=True)

            # Identify the columns that were dropped
            dropped_columns = [col for col in original_columns if col not in df_cleaned.columns]

            # Create a hashable key for the dictionary (sorted tuple of dropped columns)
            # Sorting ensures files are grouped together regardless of the order
            # columns happened to be dropped in (though usually they are consistent).
            dropped_key = tuple(sorted(dropped_columns))

            # Add the file to the list associated with this set of dropped columns
            dropped_column_groups[dropped_key].append(file)
            processed_count += 1

        except FileNotFoundError:
            error_message = "File not found"
            print(f"  ERROR: {error_message} - {file}")
            read_errors[file] = error_message
        except pd.errors.EmptyDataError:
            error_message = "File is empty"
            print(f"  ERROR: {error_message} - {file}")
            read_errors[file] = error_message
        except Exception as e:
            # Catch other potential errors during read or processing
            error_message = f"An unexpected error occurred: {e}"
            print(f"  ERROR: {error_message} - {file}")
            read_errors[file] = error_message

    print(f"\nFinished processing loop. Successfully processed: {processed_count}, Errors: {len(read_errors)}")

    # --- Report the Groups ---
    print("\n--- File Groups Based on Dropped Columns (Columns with ALL NaN values) ---")

    if not dropped_column_groups:
        print("No files were successfully processed to form groups.")
    else:
        group_num = 1
        # Iterate through the groups dictionary
        for dropped_key_tuple, file_list in dropped_column_groups.items():
            print(f"\nGroup {group_num}: {len(file_list)} file(s)")

            # Convert the tuple key back to a list for printing
            dropped_list = list(dropped_key_tuple)

            if not dropped_list:
                print("  No columns dropped (all columns had at least one non-NaN value).")
            else:
                print(f"  Columns Dropped ({len(dropped_list)}): {dropped_list}")

            # Optional: print file names, limiting for large groups
            print(f"  Files in this group:")
            if len(file_list) < 10:
                for f in file_list:
                    print(f"    - {f}")
            else:
                # Print first few and count if list is long
                for i in range(5):
                    print(f"    - {file_list[i]}")
                print(f"    - ... and {len(file_list) - 5} more")

            group_num += 1 # Increment for the next group

    # --- Report Files with Errors ---
    if read_errors:
        print("\n--- Files with Read Errors ---")
        for file, error_msg in read_errors.items():
            print(f"  {file}: {error_msg}")

    print("\nProcessing complete.")


Found 94 CSV files. Starting processing...

Finished processing loop. Successfully processed: 94, Errors: 0

--- File Groups Based on Dropped Columns (Columns with ALL NaN values) ---

Group 1: 2 file(s)
  Columns Dropped (13): ['UNNAMED', 'UNNAMED.1', 'UNNAMED.10', 'UNNAMED.11', 'UNNAMED.12', 'UNNAMED.2', 'UNNAMED.3', 'UNNAMED.4', 'UNNAMED.5', 'UNNAMED.6', 'UNNAMED.7', 'UNNAMED.8', 'UNNAMED.9']
  Files in this group:
    - Data/Ecom Owners 100k - Copy\Copy of 10.csv
    - Data/Ecom Owners 100k - Copy\Copy of 82.csv

Group 2: 42 file(s)
  No columns dropped (all columns had at least one non-NaN value).
  Files in this group:
    - Data/Ecom Owners 100k - Copy\Copy of 106.csv
    - Data/Ecom Owners 100k - Copy\Copy of 12.csv
    - Data/Ecom Owners 100k - Copy\Copy of 14.csv
    - Data/Ecom Owners 100k - Copy\Copy of 15.csv
    - Data/Ecom Owners 100k - Copy\Copy of 16.csv
    - ... and 37 more

Group 3: 20 file(s)
  Columns Dropped (1): ['UNNAMED']
  Files in this group:
    - Data/Ecom

In [ ]:
import pandas as pd
import glob
import os
import numpy as np # Import numpy (though not strictly needed for this version)
import re # Import regex module for more complex patterns if needed

# --- Configuration ---
# Make sure this path points to your CSV files
folder_file_path = "Data/Ecom Owners 100k - Copy"  # <--- Check path, ensure it's correct

# Define the rules for renaming columns based on content
# Format: { "pattern_to_find": "New_Column_Name" }
# Patterns are checked case-insensitively. The first rule that matches for a column determines its new name.
# Be mindful of pattern specificity (e.g., "youtube.com" might be too broad vs. a more specific URL structure)
RENAME_RULES = {
    "linkedin.com": "LinkedIn",
    "facebook.com": "Facebook",
    "youtube.com": "YouTube", # More specific pattern based on your example
    "instagram.com": "Instagram",
    "pinterest.com": "Pinterest",
    "tiktok.com": "TikTok",
    "Vimeo.com": "Vimeo",
    "google.com": "Google",
    "twitter.com": "Twitter",
    # Add more rules here if needed:
    # "twitter.com": "Twitter",
    # "instagram.com": "Instagram",
}
# --- End Configuration ---


if not files:
    print(f"No CSV files found in the directory: {folder_file_path}")
else:
    # Set display options for potentially wide data
    pd.set_option('display.max_rows', 10)
    pd.set_option('display.max_columns', None)
    pd.set_option('display.width', 1200)
    pd.set_option('display.max_colwidth', 50)

    for file in files:
        print(f"\n--- Processing File: {os.path.basename(file)} ---")
        try:
            df = pd.read_csv(file, low_memory=False)
            print(f"   Original shape: {df.shape}")

            # Step 1: Drop columns that are ALL NaN
            cols_before_step1 = df.columns.tolist()
            df.dropna(axis=1, how='all', inplace=True)
            cols_after_step1 = df.columns.tolist()
            dropped_step1 = [col for col in cols_before_step1 if col not in cols_after_step1]
            if dropped_step1:
                print(f"   Dropped all-NaN columns: {dropped_step1}")
            print(f"   Shape after dropping all-NaN cols: {df.shape}")

            # Check if DataFrame became empty after step 1
            if df.empty:
                print("   DataFrame is empty after dropping all-NaN columns.")
                continue # Skip to next file

            # Step 2: Identify columns with less than 5 NON-NaN values
            non_nan_counts = df.count() # df.count() gives non-NaN counts per column
            cols_to_drop_few_values = non_nan_counts[non_nan_counts < 5].index.tolist()

            # Step 3: Drop these identified columns
            if cols_to_drop_few_values:
                print(f"   Dropping columns with < 5 non-NaN values: {cols_to_drop_few_values}")
                df.drop(columns=cols_to_drop_few_values, inplace=True)
                print(f"   Shape after dropping low non-NaN count cols: {df.shape}")
            else:
                print("   No columns found with < 5 non-NaN values.")

            # Check if DataFrame became empty after step 3
            if df.empty:
                 print("   DataFrame is empty after dropping low non-NaN count columns.")
                 continue # Skip to next file

            # --- <<< MODIFIED STEP: Rename Columns Based on Content Rules START >>> ---
            rename_mapping = {} # Stores {original_col_name: new_col_name}
            processed_cols = set() # Tracks original columns already assigned a new name
            used_target_names = set() # Tracks target names already used in this file's rename_mapping

            print(f"   Checking columns for rename rules: {list(RENAME_RULES.keys())}")

            # Iterate through columns that survived the previous drops
            for col in df.columns:
                if col in processed_cols: # Skip if already decided to rename this column
                    continue

                try:
                    # Convert column to string type for reliable checking
                    col_as_str = df[col].astype(str)

                    # Check against each rule in the defined order
                    for pattern, target_name in RENAME_RULES.items():
                        # na=False treats NaN as not containing the pattern
                        # case=False makes the search case-insensitive
                        # re.escape(pattern) could be used if patterns have special regex chars, but simple contains is often fine.
                        if col_as_str.str.contains(pattern, case=False, na=False, regex=False).any():
                            # Found a match! Now check for conflicts.

                            # Conflict 1: Is this target name already assigned to another column's rename?
                            if target_name in used_target_names:
                                print(f"   WARNING: Column '{col}' matches pattern '{pattern}' for target '{target_name}', but '{target_name}' is already assigned to another column. Skipping rename for '{col}'.")
                                # Don't break here, let other patterns try for this column if needed, but likely want first match.
                                # To strictly enforce first *pattern* match wins even if target is taken, add 'break' here.
                                continue # Check next pattern for this column

                            # Conflict 2: Does a column *already exist* with the target name?
                            # (Allow renaming if the current column *already* has the target name)
                            if target_name in df.columns and col != target_name:
                                 print(f"   WARNING: Column '{col}' matches pattern '{pattern}' for target '{target_name}', but a column named '{target_name}' already exists. Skipping rename for '{col}'.")
                                 # Don't break here, let other patterns try for this column.
                                 continue # Check next pattern for this column

                            # If no conflicts, assign the rename
                            rename_mapping[col] = target_name
                            processed_cols.add(col) # Mark original column as processed
                            used_target_names.add(target_name) # Mark target name as used
                            print(f"   Match found: Column '{col}' contains '{pattern}'. Will be renamed to '{target_name}'.")
                            break # Stop checking other patterns for this column (first match wins)

                except Exception as e:
                    # Catch potential errors during string conversion/check
                    print(f"   Skipping rename check for column '{col}' due to error: {e}")
                    pass

            # Perform the renaming if any mappings were found
            if rename_mapping:
                df.rename(columns=rename_mapping, inplace=True)
                print(f"   Columns renamed based on content: {rename_mapping}")
                print(f"   Shape after renaming: {df.shape}") # Shape doesn't change
                # print(f"   Current columns: {df.columns.tolist()}") # Optional: See all columns after rename
            else:
                 print("   No columns matched the defined rename rules.")
            # --- <<< MODIFIED STEP: Rename Columns Based on Content Rules END >>> ---


            # --- Existing logic: Reconstruct DataFrame with trimmed values ---
            print(f"   Processing remaining columns for final display: {df.columns.tolist()}")
            trimmed_columns_dict = {}
            for col in df.columns: # Uses the potentially renamed columns
                original_series = df[col]
                trimmed_series = original_series.dropna()
                trimmed_columns_dict[col] = trimmed_series.reset_index(drop=True)

            result_df = pd.DataFrame(trimmed_columns_dict)

            print(f"\n   Displaying head() of DataFrame with non-NaN values shifted up:")
            if result_df.empty:
                print("     (Resulting DataFrame head is empty after shifting NaNs)")
            else:
                try:
                    from IPython.display import display
                    display(result_df.head(10))
                    result_df.to_csv(file, index=False) # Save the modified DataFrame back to CSV
                    print(f"==============================file: {file}")
                except ImportError:
                    print(result_df.head(10))

        except FileNotFoundError:
             print(f"   ERROR: File not found at specified path: {file}")
        except pd.errors.EmptyDataError:
             print(f"   ERROR: File is empty: {os.path.basename(file)}")
        except Exception as e:
             print(f"   ERROR: An unexpected error occurred processing {os.path.basename(file)}: {e}")
             import traceback
             # traceback.print_exc() # Uncomment for detailed error traceback

    # Optional: Reset display options
    # pd.reset_option('display.max_rows')
    # pd.reset_option('display.max_columns')
    # pd.reset_option('display.width')
    # pd.reset_option('display.max_colwidth')
    print("\nProcessing Complete.")

In [ ]:
import os
import pandas as pd
from column_namer_refined import generate_column_name_fewshot

"""
This script processes all csv files in a specified directory, renaming columns based on a few-shot learning model.
It uses the `generate_column_name_fewshot` function from the `column_namer_refined` module to generate new column names based on a sample of values from each column.
"""

def smart_rename_columns(files):
    for file in files:
        try:
            df = pd.read_csv(file, low_memory=False)

            print(f"File: {os.path.basename(file)}")
            print(f"  Shape: {df.shape}")

            # Show first 10 rows
            # display(df.head(10))
            print(f"  Columns: {df.columns.tolist()}")

            for col in df.columns:
                col_sample = df[col].sample(50, random_state=1).to_list()
                col_name = generate_column_name_fewshot(col_sample)  # Call the function with the sample values
                df.rename(columns={col: col_name}, inplace=True)

            print(f"  Renamed columns: {df.columns.tolist()}")

            df.to_csv(file, index=False)  # Save the modified DataFrame back to CSV

        except Exception as e:
            print(f"Error reading file {file}: {e}")


File: Copy of 10.csv
  Shape: (1315, 28)
Sample values from column 'First_Name': ['Jodine', 'Helen', 'Cole', 'Joanna', 'Ben', 'Jamie', 'Patrick', 'Michael', 'Ellie', 'Thomas', 'Jade', 'Callum', 'Yvonne', 'Laura', 'Matthew', 'Natasha', 'Jessica', 'Billy', 'Mark', 'Joseph', 'JEAN', 'Will', 'Simon', 'Lee', 'Fiona', 'Stephen', 'Daniel', 'Sandra', 'Birdie', 'Lisimba', 'Niklas', 'Olivia', 'John', 'Mark', 'Freddy', 'Kerstin', 'Adriana', 'Geoff', 'Maximillion', 'Virginie', 'Alex', 'Justin', 'James', 'Damien', 'Aimee', 'Cindy', 'Bert', 'Catherine', 'Kris', 'Luke']
First_Name
Sample values from column 'Suggested Name: Last_Name': ['Boothby', 'Pope', 'Fraser', 'Miller', 'Cleary', 'Heins', 'Cambiasso', 'Manning', 'Webb', 'Lindie', 'Nisbeck', 'Bush', 'M.', 'Handley', 'Dick', 'Whiting', 'Allen', 'Webb', 'Rogers', 'Benjamin', 'GOUPIL', 'Cochrane', 'Whitaker', 'Armstrong', 'Fawcett', 'Stephen.Jones', 'Shaw', 'Le,', 'Chapman', 'Pink', 'Oppermann', 'Rogers', 'Cussons', 'W.', 'Ward', 'Robinson', 'Gentile

In [2]:
import glob, os
import pandas as pd
for file in glob.glob(folder_file_path + "/*.csv"):
    try:
        df = pd.read_csv(file, low_memory=False)
        columns_to_drop = [col for col in df.columns if 'unnamed' in col.lower()]


        if columns_to_drop:
            # Drop the identified columns
            # axis=1 means drop columns (axis=0 means drop rows)
            # inplace=True modifies the DataFrame directly without returning a new one
            df.drop(columns=columns_to_drop, axis=1, inplace=True)
            # print(f"  New shape after dropping columns: {df.shape}")
            # print(columns_to_drop)
        if len(df.columns) < 5:
            print(f"File: {os.path.basename(file)}")
            print(df.columns.tolist())
    
    
    except Exception as e:
        print(f"Error reading file {file}")
        print(e)


File: Copy of 41.csv
['Email', 'First_Name', 'Last_Name']
File: Copy of 49.csv
['First_Name', 'Last_Name', 'Email', 'Company_Name']
File: Copy of 90.csv
['Email', 'First_Name', 'Last_Name', 'Date']
File: Copy of 94.csv
['Email', 'First_Name', 'Business_webpage', 'Date']
File: Copy of 95.csv
['Email', 'First_Name', 'Last_Name', 'Date']
File: Copy of 99.csv
['Email', 'First_Name', 'Last_Name', 'Date']


In [ ]:
import pandas as pd
import glob
from collections import defaultdict

# Find files
files = glob.glob(folder_file_path + "/*.csv")

def group_files():
    pass

if not files:
    print(f"No CSV files found in '{folder_file_path}'. Please check the path.")
else:
    print(f"Found {len(files)} CSV files. Starting processing...")

    # Dictionary to hold the groups: {tuple_of_dropped_columns: [list_of_files]}
    dropped_column_groups = defaultdict(list)
    # Dictionary to keep track of files that cause errors
    read_errors = {}
    processed_count = 0

    # --- Process Files and Group Them ---
    for file in files:
        try:
            # Minimal print during processing loop for progress indication
            # print(f"Processing: {file}")
            df = pd.read_csv(file, low_memory=False)

            original_columns = df.columns.tolist()

            # Drop columns where *ALL* values are NaN
            # Note: Your code used how='all'. If you meant *ANY* NaN, change to how='any'
            df_cleaned = df.dropna(axis=1, how='all')

            # Identify the columns that were dropped
            dropped_columns = [col for col in original_columns if col not in df_cleaned.columns]

            # Create a hashable key for the dictionary (sorted tuple of dropped columns)
            # Sorting ensures files are grouped together regardless of the order
            # columns happened to be dropped in (though usually they are consistent).
            dropped_key = tuple(sorted(dropped_columns))

            # Add the file to the list associated with this set of dropped columns
            dropped_column_groups[dropped_key].append(file)
            processed_count += 1

        except FileNotFoundError:
            error_message = "File not found"
            print(f"  ERROR: {error_message} - {file}")
            read_errors[file] = error_message
        except pd.errors.EmptyDataError:
            error_message = "File is empty"
            print(f"  ERROR: {error_message} - {file}")
            read_errors[file] = error_message
        except Exception as e:
            # Catch other potential errors during read or processing
            error_message = f"An unexpected error occurred: {e}"
            print(f"  ERROR: {error_message} - {file}")
            read_errors[file] = error_message

    print(f"\nFinished processing loop. Successfully processed: {processed_count}, Errors: {len(read_errors)}")

    # --- Report the Groups ---
    print("\n--- File Groups Based on Dropped Columns (Columns with ALL NaN values) ---")

    if not dropped_column_groups:
        print("No files were successfully processed to form groups.")
    else:
        group_num = 1
        # Iterate through the groups dictionary
        for dropped_key_tuple, file_list in dropped_column_groups.items():
            print(f"\nGroup {group_num}: {len(file_list)} file(s)")

            # Convert the tuple key back to a list for printing
            dropped_list = list(dropped_key_tuple)

            if not dropped_list:
                print("  No columns dropped (all columns had at least one non-NaN value).")
            else:
                print(f"  Columns Dropped ({len(dropped_list)}): {dropped_list}")

            # Optional: print file names, limiting for large groups
            print(f"  Files in this group:")
            if len(file_list) < 10:
                for f in file_list:
                    print(f"    - {f}")
            else:
                # Print first few and count if list is long
                for i in range(5):
                    print(f"    - {file_list[i]}")
                print(f"    - ... and {len(file_list) - 5} more")

            group_num += 1 # Increment for the next group

    # --- Report Files with Errors ---
    if read_errors:
        print("\n--- Files with Read Errors ---")
        for file, error_msg in read_errors.items():
            print(f"  {file}: {error_msg}")

    print("\nProcessing complete.")

